# Pandas & Matplotlib Basics

**Soft prerequisite** for the AI/ML course. These skills will be reinforced during Session 1 (EDA), so don't worry if you're not 100% comfortable yet.

**What this covers:** Creating and manipulating DataFrames with pandas, basic plotting with matplotlib, and combining both for quick exploratory data analysis.

**Who it's for:** Students who have used Python but have limited experience with pandas or matplotlib.

**Estimated time:** ~45 minutes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# So plots render inline in Jupyter
%matplotlib inline

---
## Dataset

We'll use a small inline dataset throughout this notebook: exam scores for 20 students.

In [ ]:
np.random.seed(42)

data = {
    "student_id": list(range(1, 21)),
    "name": ["Alice", "Bob", "Carlos", "Diana", "Eve",
             "Frank", "Grace", "Hector", "Ivy", "Jack",
             "Karen", "Leo", "Mona", "Nate", "Olivia",
             "Paul", "Quinn", "Rosa", "Sam", "Tina"],
    "major": ["CS", "CS", "Math", "CS", "Math",
              "Physics", "CS", "Math", "Physics", "CS",
              "Math", "Physics", "CS", "CS", "Math",
              "Physics", "CS", "Math", "Physics", "CS"],
    "study_hours": [10, 5, 8, 12, 7, 3, 15, 9, 6, 11,
                    4, 8, 14, 2, 10, 7, 13, np.nan, 5, 9],
    "midterm_score": [85, 62, 78, 91, 70, 55, 95, 80, 65, 88,
                      np.nan, 72, 93, 48, 82, 68, 90, 76, 60, 84],
    "final_score": [88, 65, 80, 94, 73, 58, 97, 82, 68, 90,
                    56, 75, 95, 50, 85, 70, 92, 79, np.nan, 86],
}

df = pd.DataFrame(data)
print(f"Shape: {df.shape}")
df.head()

---
## 1. Series and DataFrame Creation

A **Series** is a 1D labeled array. A **DataFrame** is a 2D table made of multiple Series (columns).

You can create them from lists, dicts, or NumPy arrays.

In [ ]:
# Series from a list
scores = pd.Series([85, 62, 78, 91], index=["Alice", "Bob", "Carlos", "Diana"], name="midterm")
print(scores)
print(f"\nType: {type(scores)}")

In [ ]:
# DataFrame from a dict of lists
small_df = pd.DataFrame({
    "name": ["Alice", "Bob"],
    "score": [85, 62],
})
small_df

### Exercise 1.1

Create a Series called `hours` with values `[3, 7, 12, 5]` and index `["Mon", "Tue", "Wed", "Thu"]`. Print it.

In [ ]:
# YOUR CODE HERE

### Exercise 1.2

Create a DataFrame with 3 columns (`city`, `population`, `country`) and at least 3 rows. Display it.

In [ ]:
# YOUR CODE HERE

---
## 2. Reading CSV Files

`pd.read_csv()` is the most common way to load data. Key parameters:
- `sep` (delimiter, default `','`)
- `header` (row number for column names)
- `index_col` (column to use as index)
- `na_values` (additional strings to treat as NaN)

Since we're keeping this notebook self-contained, we'll simulate a CSV with `io.StringIO`.

In [ ]:
from io import StringIO

csv_text = """name,age,grade
Alice,22,A
Bob,25,B
Carlos,23,A
"""

df_csv = pd.read_csv(StringIO(csv_text))
print(df_csv)
print(f"\nColumn types:\n{df_csv.dtypes}")

In practice you'll do `pd.read_csv("path/to/file.csv")`. Other useful readers: `pd.read_excel()`, `pd.read_json()`, `pd.read_parquet()`.

### Exercise 2.1

Use `StringIO` and `pd.read_csv` to load this CSV string into a DataFrame and display it:

```
product,price,quantity
apple,1.2,50
banana,0.5,120
cherry,3.0,30
```

In [ ]:
# YOUR CODE HERE

---
## 3. Indexing, Filtering, Selecting Columns

Three main ways to select data:

| Method | Syntax | Use |
|--------|--------|-----|
| `[]` | `df["col"]` or `df[["col1","col2"]]` | Select columns |
| `.loc[]` | `df.loc[row_label, col_label]` | Label-based |
| `.iloc[]` | `df.iloc[row_idx, col_idx]` | Position-based |

**Filtering** uses boolean masks: `df[df["col"] > value]`.

In [ ]:
# Select a single column (returns Series)
print(df["name"].head())

# Select multiple columns (returns DataFrame)
print()
print(df[["name", "midterm_score"]].head())

In [ ]:
# Filter: students with midterm > 80
high_scorers = df[df["midterm_score"] > 80]
print(f"Students with midterm > 80: {len(high_scorers)}")
print(high_scorers[["name", "midterm_score"]])

In [ ]:
# Combine filters with & (and) or | (or) — use parentheses!
cs_high = df[(df["major"] == "CS") & (df["midterm_score"] > 80)]
print(cs_high[["name", "major", "midterm_score"]])

### Exercise 3.1

Select all Physics majors and display their `name`, `study_hours`, and `final_score`.

In [ ]:
# YOUR CODE HERE

### Exercise 3.2

Select students who studied more than 10 hours AND scored above 85 on the final. Display their names.

In [ ]:
# YOUR CODE HERE

---
## 4. Basic Operations: groupby, describe, value_counts, apply

These are your go-to tools for quick data exploration.

In [ ]:
# describe() — summary statistics for numeric columns
df.describe()

In [ ]:
# value_counts() — frequency of each unique value
print(df["major"].value_counts())

In [ ]:
# groupby() — split-apply-combine
avg_by_major = df.groupby("major")[["midterm_score", "final_score"]].mean()
print(avg_by_major)

In [ ]:
# apply() — apply a function to each element or row
# Example: letter grade based on final score
def letter_grade(score):
    if pd.isna(score):
        return "N/A"
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    elif score >= 60:
        return "D"
    return "F"

df["grade"] = df["final_score"].apply(letter_grade)
print(df[["name", "final_score", "grade"]].head(10))

### Exercise 4.1

Use `groupby` to find the **maximum** `study_hours` per major.

In [ ]:
# YOUR CODE HERE

### Exercise 4.2

Use `apply` to create a new column `passed` that is `True` if `final_score >= 60`, `False` otherwise. Then use `value_counts()` on that column.

In [ ]:
# YOUR CODE HERE

---
## 5. Handling Missing Values

Real datasets have missing values. Pandas represents them as `NaN` (Not a Number).

| Method | Purpose |
|--------|---------|
| `df.isna()` | Boolean mask of missing values |
| `df.isna().sum()` | Count missing per column |
| `df.dropna()` | Drop rows with any NaN |
| `df.fillna(value)` | Replace NaN with a value |

In [ ]:
# Check for missing values
print(df.isna().sum())
print(f"\nTotal missing: {df.isna().sum().sum()}")

In [ ]:
# Option 1: Drop rows with any missing value
df_clean = df.dropna()
print(f"Before dropna: {len(df)} rows")
print(f"After dropna:  {len(df_clean)} rows")

In [ ]:
# Option 2: Fill missing values (e.g., with the column median)
df_filled = df.copy()
df_filled["study_hours"] = df_filled["study_hours"].fillna(df_filled["study_hours"].median())
df_filled["midterm_score"] = df_filled["midterm_score"].fillna(df_filled["midterm_score"].median())
df_filled["final_score"] = df_filled["final_score"].fillna(df_filled["final_score"].median())
print(f"Missing after fillna: {df_filled.isna().sum().sum()}")

### Exercise 5.1

Which rows have missing data? Display only the rows from `df` that contain at least one NaN. Hint: use `df[df.isna().any(axis=1)]`.

In [ ]:
# YOUR CODE HERE

---
## 6. Matplotlib: Line Plots, Scatter Plots, Histograms

matplotlib.pyplot provides a MATLAB-like plotting interface. The three most common plot types for EDA:

In [ ]:
# Line plot — good for trends over ordered data
plt.figure(figsize=(8, 4))
plt.plot(df["student_id"], df["midterm_score"], marker="o", label="Midterm")
plt.plot(df["student_id"], df["final_score"], marker="s", label="Final")
plt.xlabel("Student ID")
plt.ylabel("Score")
plt.title("Exam Scores by Student")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot — good for relationships between two variables
plt.figure(figsize=(6, 5))
plt.scatter(df["study_hours"], df["final_score"], c="steelblue", edgecolors="black", alpha=0.7)
plt.xlabel("Study Hours")
plt.ylabel("Final Score")
plt.title("Study Hours vs Final Score")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Histogram — good for understanding distributions
plt.figure(figsize=(6, 4))
plt.hist(df["final_score"].dropna(), bins=8, color="steelblue", edgecolor="black", alpha=0.7)
plt.xlabel("Final Score")
plt.ylabel("Count")
plt.title("Distribution of Final Scores")
plt.tight_layout()
plt.show()

### Exercise 6.1

Create a scatter plot of `midterm_score` (x-axis) vs `final_score` (y-axis). Add axis labels and a title.

In [ ]:
# YOUR CODE HERE

### Exercise 6.2

Create a histogram of `study_hours`. Choose an appropriate number of bins.

In [ ]:
# YOUR CODE HERE

---
## 7. Labels, Titles, Legends, Subplots

Good plots need clear labels. Use `plt.subplots()` to create multi-panel figures.

In [ ]:
# Subplots: 1 row, 2 columns
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left panel: histogram of midterm
axes[0].hist(df["midterm_score"].dropna(), bins=8, color="coral", edgecolor="black")
axes[0].set_xlabel("Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Midterm Distribution")

# Right panel: histogram of final
axes[1].hist(df["final_score"].dropna(), bins=8, color="steelblue", edgecolor="black")
axes[1].set_xlabel("Score")
axes[1].set_ylabel("Count")
axes[1].set_title("Final Distribution")

fig.suptitle("Score Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Exercise 7.1

Create a figure with 1 row and 3 columns. In each subplot, show a scatter plot of `study_hours` vs `final_score` for one major (CS, Math, Physics). Title each subplot with the major name.

In [ ]:
# YOUR CODE HERE

---
## 8. Combining Pandas + Matplotlib for Quick EDA

Pandas has built-in `.plot()` methods that wrap matplotlib. This is the fastest way to explore data.

In [ ]:
# Bar plot of average scores by major (directly from groupby)
df.groupby("major")[["midterm_score", "final_score"]].mean().plot.bar(figsize=(7, 4), rot=0)
plt.ylabel("Average Score")
plt.title("Average Scores by Major")
plt.legend(["Midterm", "Final"])
plt.tight_layout()
plt.show()

In [ ]:
# Box plot — great for comparing distributions across groups
df.boxplot(column="final_score", by="major", figsize=(6, 4))
plt.suptitle("")  # remove auto-generated title
plt.title("Final Score by Major")
plt.ylabel("Score")
plt.tight_layout()
plt.show()

In [ ]:
# Quick EDA workflow:
# 1. Shape and types
print(f"Shape: {df.shape}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nMissing:\n{df.isna().sum()}")
print(f"\nNumeric summary:")
df.describe()

### Exercise 8.1

Create a bar plot showing the count of students per major. Use `value_counts()` and `.plot.bar()`.

In [ ]:
# YOUR CODE HERE

### Exercise 8.2

Perform a mini-EDA on the dataset:
1. Print the shape, dtypes, and missing value counts
2. Show `describe()` for numeric columns
3. Create one plot that you think best summarizes the data

There is no single correct answer here — the goal is to practice the workflow.

In [ ]:
# YOUR CODE HERE

---
## Self-Assessment

Answer these without looking back at the notebook. If you can answer 3 out of 4, you're in good shape.

1. What is the difference between `df.loc[]` and `df.iloc[]`?

2. How would you filter a DataFrame to get rows where column `age` is between 20 and 30 (inclusive)?

3. You have a DataFrame with 5% missing values in a column. Name two strategies for handling them and when you'd choose each.

4. What is the difference between `plt.plot()` and `df.plot()`? When would you use each?

---
## References

- [pandas Getting Started Tutorials](https://pandas.pydata.org/docs/getting_started/intro_tutorials/) -- Official step-by-step guides covering all the basics.
- [Matplotlib Tutorials](https://matplotlib.org/stable/tutorials/index.html) -- Official tutorials from basic to advanced plotting.
- [Python Data Science Handbook (Jake VanderPlas)](https://jakevdp.github.io/PythonDataScienceHandbook/) -- Free online book; chapters 3 (pandas) and 4 (matplotlib) are particularly relevant.